<a href="https://colab.research.google.com/github/SRIJANRAOS/srijanraos_INFO5731_spring2026/blob/main/srijanraos_Assignment_2_final_(1)_2_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# **INFO5731 Assignment 2**

In this assignment, we will delve into various aspects of natural language processing (NLP) and text analysis. The tasks are designed to deepen your understanding of key NLP concepts and techniques, as well as to provide hands-on experience with practical applications.

Through these tasks, you'll gain practical experience in NLP techniques such as N-gram analysis, TF-IDF, word embedding model creation, and sentiment analysis dataset creation.

**Expectations**:
* Use the provided `.ipynb` document to write your code and respond to the questions. Avoid generating a new file.
* Write complete answers and run all the cells before submission.
* Make sure the submission is "clean"; i.e., no unnecessary code cells.
* Once finished, allow shared rights from the top right corner (see Canvas for details).
* **Note:** Use the same dataset you created in **Assignment 1** for **Questions 1–3**.

**Total points:** 100

**Deadline:** See Canvas

Late submission will have a penalty of **10% reduction for each day** after the deadline.


## Question 1 (25 points)

**Understand N-gram**

Write a **Python** program to conduct N-gram analysis based on the dataset you created in **Assignment 1**. You need to write **code from scratch instead of using any pre-existing libraries** to do so:

(1) Count the frequency of all the N-grams (**N = 3** and **N = 2**).

(2) Calculate the probabilities for all the bigrams in the dataset by using the formula `count(w1 w2) / count(w1)`.

For example, `count(really like) / count(really) = 1 / 3 = 0.33`.

(3) Extract all the noun phrases and calculate the relative probabilities of each review in terms of other reviews (abstracts, or tweets) by using the formula `frequency(noun phrase) / max frequency(noun phrase)` on the whole dataset. You may use NLP libraries (e.g., **spaCy** or **NLTK**) for noun phrase extraction.

Print out the result in a table with all noun phrases as the column names and all **100** reviews (abstracts, or tweets) as the row names.


In [2]:
"""
N-gram Analysis on Semantic Scholar Abstracts
Assignment 1 – Question 1
=============================================================
(1) Count frequencies of all trigrams (N=3) and bigrams (N=2).
(2) Calculate bigram probabilities: count(w1 w2) / count(w1).
(3) Extract noun phrases and compute relative probabilities:
    frequency(NP in doc) / max_frequency(NP across all docs).
    Results printed as a table: rows = abstracts, columns = NPs.

HOW TO RUN IN GOOGLE COLAB / JUPYTER:
  Option A – upload via Colab file panel, then set:
      CSV_PATH = "/content/semantic_scholar_raw_abstracts.csv"
  Option B – upload inline, then set:
      CSV_PATH = "semantic_scholar_raw_abstracts.csv"
  Option C – use the file picker cell below (uncomment it).
"""

# ── Uncomment the next 5 lines if running in Google Colab ──────────────────
# from google.colab import files
# uploaded = files.upload()          # opens a file-picker dialog
# import io
# CSV_PATH = list(uploaded.keys())[0]
# ---------------------------------------------------------------------------

import re
import csv
import math
from collections import defaultdict, Counter
import pandas as pd

# ── NLP tools (no data download required) ─────────────────────────────────────
from nltk.tag import RegexpTagger
from nltk import RegexpParser

# ── 1.  Load dataset (first 100 non-null abstracts) ───────────────────────────
# ▶▶ CHANGE THIS PATH to wherever you saved the CSV ◀◀
# Examples:
#   Google Colab after upload:  "/content/semantic_scholar_raw_abstracts.csv"
#   Jupyter same folder:        "semantic_scholar_raw_abstracts.csv"
#   Windows full path:          r"C:\Users\you\Downloads\semantic_scholar_raw_abstracts.csv"
CSV_PATH = "semantic_scholar_raw_abstracts.csv"   # ← edit this line

df_full = pd.read_csv(CSV_PATH)
df = df_full[df_full["abstract"].notna()].head(100).reset_index(drop=True)
abstracts = df["abstract"].tolist()
titles    = [f"Abstract_{i+1}" for i in range(len(abstracts))]

print(f"Loaded {len(abstracts)} abstracts.\n")

# ── 2.  Tokeniser (no external data required) ─────────────────────────────────

def tokenize(text: str) -> list[str]:
    """Lowercase and split on non-alphanumeric characters."""
    text = text.lower()
    tokens = re.findall(r"\b[a-z][a-z'\-]*[a-z]\b|\b[a-z]\b", text)
    return tokens

STOPWORDS = {
    "a","an","the","and","or","but","in","on","at","to","for","of","with",
    "by","from","as","is","are","was","were","be","been","being","have",
    "has","had","that","this","these","those","it","its","we","our","they",
    "their","which","also","can","may","will","more","been","not","no",
    "into","than","each","all","both","other","such","after","before","up",
    "out","over","under","between","through","about","while","when","where",
    "how","what","if","so","do","does","did","he","she","his","her","i",
    "you","your","them","us","my","me","who","there","here","any","some",
    "most","very","just","only","then","than","thus","yet","nor"
}

# ── 3.  Build corpus token lists ──────────────────────────────────────────────

corpus_tokens: list[list[str]] = [tokenize(a) for a in abstracts]

# ── ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ ──
# PART (1) – N-gram frequency counts  (from scratch, no library)
# ── ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ ──

def build_ngrams(tokens: list[str], n: int) -> list[tuple]:
    """Return all n-grams from a token list as tuples."""
    return [tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]

# Aggregate across all 100 abstracts
bigram_counts:  Counter = Counter()
trigram_counts: Counter = Counter()

for tokens in corpus_tokens:
    bigram_counts.update(build_ngrams(tokens, 2))
    trigram_counts.update(build_ngrams(tokens, 3))

print("=" * 70)
print("PART (1) — N-gram Frequency Counts")
print("=" * 70)

print(f"\nTotal unique bigrams  (N=2): {len(bigram_counts):,}")
print(f"Total unique trigrams (N=3): {len(trigram_counts):,}")

TOP_N = 20

print(f"\nTop {TOP_N} Bigrams (N=2) by frequency:")
print(f"{'Rank':<6} {'Bigram':<40} {'Count':>6}")
print("-" * 55)
for rank, (ngram, cnt) in enumerate(bigram_counts.most_common(TOP_N), 1):
    print(f"{rank:<6} {' '.join(ngram):<40} {cnt:>6}")

print(f"\nTop {TOP_N} Trigrams (N=3) by frequency:")
print(f"{'Rank':<6} {'Trigram':<55} {'Count':>6}")
print("-" * 68)
for rank, (ngram, cnt) in enumerate(trigram_counts.most_common(TOP_N), 1):
    print(f"{rank:<6} {' '.join(ngram):<55} {cnt:>6}")


# ── ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ ──
# PART (2) – Bigram probabilities: count(w1 w2) / count(w1)
# ── ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ ──

# Unigram counts (needed as denominator)
unigram_counts: Counter = Counter()
for tokens in corpus_tokens:
    unigram_counts.update(tokens)

def bigram_probability(w1: str, w2: str) -> float:
    """P(w2 | w1) = count(w1 w2) / count(w1)."""
    denom = unigram_counts.get(w1, 0)
    if denom == 0:
        return 0.0
    return bigram_counts.get((w1, w2), 0) / denom

print("\n" + "=" * 70)
print("PART (2) — Bigram Probabilities  [count(w1 w2) / count(w1)]")
print("=" * 70)

# Show top 20 bigrams by probability (require count ≥ 5 for stability)
stable_bigrams = [(bg, cnt) for bg, cnt in bigram_counts.items() if cnt >= 5]
bigram_probs = [
    (bg, cnt, bigram_probability(bg[0], bg[1]))
    for bg, cnt in stable_bigrams
]
bigram_probs.sort(key=lambda x: -x[2])

print(f"\nTop {TOP_N} Bigrams by probability (min frequency 5):")
print(f"{'Rank':<6} {'w1':<20} {'w2':<20} {'count(w1 w2)':>12} {'count(w1)':>10} {'P(w2|w1)':>10}")
print("-" * 80)
for rank, (bg, cnt, prob) in enumerate(bigram_probs[:TOP_N], 1):
    w1, w2 = bg
    print(f"{rank:<6} {w1:<20} {w2:<20} {cnt:>12} {unigram_counts[w1]:>10} {prob:>10.4f}")

print(f"\nAll {len(bigram_probs)} stable bigrams (count ≥ 5) probabilities written to Part2 table above.")
print("Full lookup function bigram_probability(w1, w2) is available programmatically.")


# ── ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ ──
# PART (3) – Noun phrase extraction + relative probability table
# ── ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ ──

# POS tagger using regex rules (no corpus download required)
# Patterns ordered from most specific to most general
POS_PATTERNS = [
    # Common adverbs
    (r".*ly$",       "RB"),
    # Adjectives: comparative / superlative
    (r".*er$",       "JJR"),
    (r".*est$",      "JJS"),
    # Verbs: participles / past tense
    (r".*ing$",      "VBG"),
    (r".*ed$",       "VBD"),
    # Determiners / articles
    (r"^(a|an|the)$", "DT"),
    # Common modal verbs
    (r"^(can|could|may|might|will|would|shall|should|must)$", "MD"),
    # Common prepositions → IN
    (r"^(in|on|at|by|for|with|of|to|from|as|into|onto|upon|"
     r"through|throughout|during|within|among|between|against|"
     r"across|along|behind|beyond|below|above|under|over|after|"
     r"before|since|until|about|around|near|toward|towards|per)$", "IN"),
    # Common conjunctions
    (r"^(and|or|but|nor|so|yet|both|either|neither|whether|if|"
     r"although|because|since|while|when|where|that|which|who)$", "CC"),
    # Common verbs (simple heuristic: short, common verb forms)
    (r"^(is|are|was|were|be|been|being|have|has|had|do|does|did|"
     r"make|makes|made|use|uses|used|show|shows|showed|shown|"
     r"provide|provides|provided|improve|improves|improved|"
     r"increase|increases|increased|reduce|reduces|reduced|"
     r"achieve|achieves|achieved|enable|enables|enabled|"
     r"propose|proposes|proposed|present|presents|presented|"
     r"develop|develops|developed|apply|applies|applied|"
     r"demonstrate|demonstrates|demonstrated|evaluate|evaluates|"
     r"evaluated|analyze|analyzes|analyzed|study|studies|studied|"
     r"investigate|investigates|investigated|compare|compares|"
     r"compared|consider|considers|considered|require|requires|"
     r"required|include|includes|included|contain|contains|"
     r"contained|allow|allows|allowed|yield|yields|yielded|"
     r"obtain|obtains|obtained|find|finds|found|show|shows|"
     r"give|gives|gave|given|take|takes|took|taken|become|"
     r"becomes|became|come|comes|came|go|goes|went|get|gets|"
     r"got|set|sets|put|puts|run|runs|ran|work|works|worked|"
     r"based|learn|learns|learned|train|trains|trained)$", "VB"),
    # Plural nouns (ends in -s)
    (r".*s$",        "NNS"),
    # Adjective-like: ends in -al, -ic, -ive, -ous, -ful, -less, -able, -ible, -ary
    (r".*(?:al|ic|ive|ous|ful|less|able|ible|ary|ary|ent|ant)$", "JJ"),
    # Numbers
    (r"^-?[0-9]+(?:\.[0-9]+)?$", "CD"),
    # Default: assume noun
    (r".*",          "NN"),
]

pos_tagger = RegexpTagger(POS_PATTERNS)

# Noun-phrase grammar: optional determiner, optional adjectives, one or more nouns
NP_GRAMMAR = r"""
    NP: {<DT>?<JJ.*>*<NN.*>+}
"""
np_parser = RegexpParser(NP_GRAMMAR)


def extract_noun_phrases(text: str) -> list[str]:
    """Return a list of noun phrase strings from text."""
    # Use original-case words for NP extraction, then lowercase result
    words = re.findall(r"\b[a-zA-Z][a-zA-Z'\-]*[a-zA-Z]\b|\b[a-zA-Z]\b", text.lower())
    # Remove stopwords only at the start/end of the phrase (not internally)
    tagged = pos_tagger.tag(words)
    tree = np_parser.parse(tagged)
    phrases = []
    for subtree in tree.subtrees(filter=lambda t: t.label() == "NP"):
        phrase_words = [w for w, _ in subtree.leaves()]
        # Drop NPs that are purely stopwords
        content = [w for w in phrase_words if w not in STOPWORDS]
        if not content:
            continue
        # Strip leading/trailing stopwords
        while phrase_words and phrase_words[0] in STOPWORDS:
            phrase_words.pop(0)
        while phrase_words and phrase_words[-1] in STOPWORDS:
            phrase_words.pop()
        if phrase_words:
            phrases.append(" ".join(phrase_words))
    return phrases


print("\n" + "=" * 70)
print("PART (3) — Noun Phrase Relative Probability Table")
print("=" * 70)
print("Extracting noun phrases from 100 abstracts …")

# ── Extract NPs per document ──────────────────────────────────────────────────
doc_np_counts: list[Counter] = []
all_nps_global: Counter = Counter()   # frequency across ALL docs

for i, text in enumerate(abstracts):
    np_list = extract_noun_phrases(text)
    c = Counter(np_list)
    doc_np_counts.append(c)
    all_nps_global.update(c)          # raw occurrence count per doc
    if (i + 1) % 25 == 0:
        print(f"  … processed {i+1}/100")

print(f"  Done. Total unique noun phrases: {len(all_nps_global):,}")

# ── Keep the top-K most frequent NPs to keep the table readable ───────────────
TOP_NP = 50  # columns
top_nps = [np for np, _ in all_nps_global.most_common(TOP_NP)]

# ── Compute max frequency for each NP across all documents ───────────────────
max_freq: dict[str, int] = {}
for np_str in top_nps:
    max_freq[np_str] = max(c.get(np_str, 0) for c in doc_np_counts)

# ── Build relative probability table ─────────────────────────────────────────
# rel_prob[doc][np] = freq(np in doc) / max_freq(np across all docs)
rows = []
for i, c in enumerate(doc_np_counts):
    row = {"Document": titles[i]}
    for np_str in top_nps:
        freq_in_doc = c.get(np_str, 0)
        mf = max_freq[np_str]
        row[np_str] = round(freq_in_doc / mf, 4) if mf > 0 else 0.0
    rows.append(row)

rel_prob_df = pd.DataFrame(rows).set_index("Document")

# ── Print the table (truncated for readability) ───────────────────────────────
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.max_rows", 110)

print(f"\nRelative Probability Table  (rows: 100 abstracts | cols: top {TOP_NP} NPs)")
print(f"Formula: freq(NP in doc) / max_freq(NP across all docs)\n")
print(rel_prob_df.to_string())

# ── Save outputs ──────────────────────────────────────────────────────────────
import os
OUT_DIR = "ngram_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

# Save full NP table to CSV
OUT_TABLE = os.path.join(OUT_DIR, "np_relative_probability_table.csv")
rel_prob_df.to_csv(OUT_TABLE)

# Save bigram probability table to CSV
bigram_prob_rows = []
for bg, cnt, prob in bigram_probs:
    bigram_prob_rows.append({
        "w1": bg[0], "w2": bg[1],
        "bigram": f"{bg[0]} {bg[1]}",
        "count_bigram": cnt,
        "count_w1": unigram_counts[bg[0]],
        "probability_P(w2|w1)": round(prob, 6)
    })
bigram_prob_df = pd.DataFrame(bigram_prob_rows)
OUT_BIGRAM = os.path.join(OUT_DIR, "bigram_probabilities.csv")
bigram_prob_df.to_csv(OUT_BIGRAM, index=False)

# Save top N-gram counts to CSV
top_bigram_rows = [{"bigram": " ".join(bg), "count": cnt}
                   for bg, cnt in bigram_counts.most_common()]
top_trigram_rows = [{"trigram": " ".join(tg), "count": cnt}
                    for tg, cnt in trigram_counts.most_common()]
pd.DataFrame(top_bigram_rows).to_csv(os.path.join(OUT_DIR, "bigram_counts.csv"), index=False)
pd.DataFrame(top_trigram_rows).to_csv(os.path.join(OUT_DIR, "trigram_counts.csv"), index=False)

print("\n\nFiles saved to folder:", OUT_DIR)
print(f"  {OUT_TABLE}")
print(f"  {OUT_BIGRAM}")
print(f"  {OUT_DIR}/bigram_counts.csv")
print(f"  {OUT_DIR}/trigram_counts.csv")


Loaded 100 abstracts.

PART (1) — N-gram Frequency Counts

Total unique bigrams  (N=2): 16,501
Total unique trigrams (N=3): 20,798

Top 20 Bigrams (N=2) by frequency:
Rank   Bigram                                    Count
-------------------------------------------------------
1      of the                                      129
2      machine learning                            108
3      in the                                       91
4      and the                                      47
5      based on                                     46
6      on the                                       41
7      in this                                      39
8      with the                                     36
9      this paper                                   31
10     this study                                   30
11     deep learning                                29
12     to the                                       28
13     such as                                      28
14     

## Question 2 (25 points)

**Understand TF-IDF and Document Representation**

Starting from the documents (all the reviews, abstracts, or tweets) collected for **Assignment 1**, write a **Python** program:

(1) Build the **document-term weight (`tf * idf`) matrix**.

(2) Rank the documents with respect to a query (design a query by yourself, for example, "An outstanding movie with a haunting performance and best character development") by using cosine similarity.

**Note:** You need to write **code from scratch instead of using any pre-existing libraries** to do so.


In [3]:
"""
TF-IDF and Document Representation
Assignment – Question 2
================================================================
(1) Build the document-term TF*IDF weight matrix from scratch.
(2) Rank all 100 documents against a custom query using cosine
    similarity (also computed from scratch).

NO sklearn / scipy / gensim or similar libraries used.
Only pandas (for display) and math/collections from stdlib.
================================================================

HOW TO RUN IN GOOGLE COLAB / JUPYTER:
  Set CSV_PATH to your uploaded file location, e.g.:
      CSV_PATH = "/content/semantic_scholar_raw_abstracts.csv"  # Colab
      CSV_PATH = "semantic_scholar_raw_abstracts.csv"            # Jupyter
"""

import re
import math
import os
from collections import Counter, defaultdict
import pandas as pd

# ── CONFIG ────────────────────────────────────────────────────────────────────
# ▶▶ CHANGE THIS PATH to wherever you saved the CSV ◀◀
CSV_PATH = "semantic_scholar_raw_abstracts.csv"

# Query designed for a machine-learning / AI research corpus
QUERY = (
    "deep learning model for accurate prediction and classification "
    "using neural network algorithms"
)

TOP_K   = 20   # how many top-ranked docs to display
TOP_TF_IDF_TERMS = 10   # top terms to show per doc in the summary
# ─────────────────────────────────────────────────────────────────────────────

STOPWORDS = {
    "a","an","the","and","or","but","in","on","at","to","for","of","with",
    "by","from","as","is","are","was","were","be","been","being","have",
    "has","had","that","this","these","those","it","its","we","our","they",
    "their","which","also","can","may","will","more","not","no","into",
    "than","each","all","both","other","such","after","before","up","out",
    "over","under","between","through","about","while","when","where","how",
    "what","if","so","do","does","did","he","she","his","her","i","you",
    "your","them","us","my","me","who","there","here","any","some","most",
    "very","just","only","then","nor","via","per","et","al","de","en",
    "within","without","among","however","therefore","thus","hence","due",
    "using","used","use","uses","show","shows","shown","paper","study",
    "studies","work","results","result","based","proposed","approach",
    "system","method","methods","two","three","one","well","new","high",
    "low","large","small","different","various","several","many","much",
    "number","order","given","found","known","make","makes","need","needs",
}


# ══════════════════════════════════════════════════════════════════════════════
#  STEP 0 – Load data
# ══════════════════════════════════════════════════════════════════════════════

df_full = pd.read_csv(CSV_PATH)
df = df_full[df_full["abstract"].notna()].head(100).reset_index(drop=True)
raw_docs   = df["abstract"].tolist()
doc_labels = [f"Abstract_{i+1:03d}" for i in range(len(raw_docs))]
N = len(raw_docs)          # number of documents
print(f"Loaded {N} abstracts.\n")


# ══════════════════════════════════════════════════════════════════════════════
#  STEP 1 – Tokeniser (from scratch, no NLTK)
# ══════════════════════════════════════════════════════════════════════════════

def tokenize(text: str) -> list[str]:
    """
    Lowercase → strip punctuation → split → remove stopwords & short tokens.
    Returns a list of content-bearing word tokens.
    """
    text = text.lower()
    # keep only alphabetic characters and spaces
    text = re.sub(r"[^a-z\s]", " ", text)
    tokens = text.split()
    return [t for t in tokens if t not in STOPWORDS and len(t) > 2]


# Tokenise every document once; store token lists
doc_tokens: list[list[str]] = [tokenize(doc) for doc in raw_docs]


# ══════════════════════════════════════════════════════════════════════════════
#  STEP 2 – Build vocabulary
# ══════════════════════════════════════════════════════════════════════════════

vocab_set: set[str] = set()
for tokens in doc_tokens:
    vocab_set.update(tokens)

vocab: list[str] = sorted(vocab_set)          # deterministic ordering
term_to_idx: dict[str, int] = {t: i for i, t in enumerate(vocab)}
V = len(vocab)
print(f"Vocabulary size : {V:,} unique terms")


# ══════════════════════════════════════════════════════════════════════════════
#  STEP 3 – Term Frequency (TF)
# ══════════════════════════════════════════════════════════════════════════════
#
#  We use *normalised* TF so that longer documents are not unfairly
#  boosted:
#
#      TF(t, d) = count(t in d) / total_tokens(d)
#
# ══════════════════════════════════════════════════════════════════════════════

# tf_matrix[doc_idx][term] = normalised TF value
tf_matrix: list[dict[str, float]] = []

for tokens in doc_tokens:
    counts = Counter(tokens)
    total  = len(tokens) if tokens else 1        # avoid div-by-zero
    tf_matrix.append({term: cnt / total for term, cnt in counts.items()})


# ══════════════════════════════════════════════════════════════════════════════
#  STEP 4 – Inverse Document Frequency (IDF)
# ══════════════════════════════════════════════════════════════════════════════
#
#  Standard smoothed IDF:
#
#      IDF(t) = log( (N + 1) / (df(t) + 1) ) + 1
#
#  where df(t) = number of documents that contain term t.
#  The +1 inside and outside the log avoids zero-division and ensures
#  very common terms still receive a small positive weight.
#
# ══════════════════════════════════════════════════════════════════════════════

# Document frequency: how many docs contain each term
df_counts: dict[str, int] = defaultdict(int)
for tf_row in tf_matrix:
    for term in tf_row:
        df_counts[term] += 1

idf: dict[str, float] = {}
for term in vocab:
    idf[term] = math.log((N + 1) / (df_counts[term] + 1)) + 1


# ══════════════════════════════════════════════════════════════════════════════
#  STEP 5 – TF-IDF matrix
# ══════════════════════════════════════════════════════════════════════════════
#
#  tfidf_matrix[doc_idx][term] = TF(t,d) * IDF(t)
#
# ══════════════════════════════════════════════════════════════════════════════

tfidf_matrix: list[dict[str, float]] = []
for tf_row in tf_matrix:
    tfidf_row = {term: tf_val * idf[term] for term, tf_val in tf_row.items()}
    tfidf_matrix.append(tfidf_row)


# ── Convert to a dense pandas DataFrame for display ──────────────────────────
print("Building dense TF-IDF DataFrame …")
tfidf_df = pd.DataFrame(
    [
        {term: row.get(term, 0.0) for term in vocab}
        for row in tfidf_matrix
    ],
    index=doc_labels,
)
tfidf_df.index.name = "Document"
print(f"TF-IDF matrix shape: {tfidf_df.shape}  (docs × terms)\n")


# ══════════════════════════════════════════════════════════════════════════════
#  STEP 6 – Cosine Similarity (from scratch)
# ══════════════════════════════════════════════════════════════════════════════
#
#  cosine_sim(A, B) = dot(A, B) / (||A|| * ||B||)
#
#  Both document and query vectors live in the same V-dimensional
#  TF-IDF space.
#
# ══════════════════════════════════════════════════════════════════════════════

def dot_product(vec_a: dict[str, float], vec_b: dict[str, float]) -> float:
    """Sparse dot product over shared keys."""
    # iterate over the shorter vector for efficiency
    if len(vec_a) > len(vec_b):
        vec_a, vec_b = vec_b, vec_a
    return sum(vec_a[t] * vec_b.get(t, 0.0) for t in vec_a)


def vector_norm(vec: dict[str, float]) -> float:
    """Euclidean (L2) norm of a sparse vector."""
    return math.sqrt(sum(v * v for v in vec.values()))


def cosine_similarity(vec_a: dict[str, float],
                      vec_b: dict[str, float]) -> float:
    """
    Cosine similarity between two sparse TF-IDF vectors.
    Returns 0.0 when either vector is the zero vector.
    """
    norm_a = vector_norm(vec_a)
    norm_b = vector_norm(vec_b)
    if norm_a == 0.0 or norm_b == 0.0:
        return 0.0
    return dot_product(vec_a, vec_b) / (norm_a * norm_b)


# ══════════════════════════════════════════════════════════════════════════════
#  STEP 7 – Query vectorisation
# ══════════════════════════════════════════════════════════════════════════════
#
#  The query is treated as a mini-document:
#   1. Tokenise with the same function.
#   2. Compute TF (raw counts / query length).
#   3. Apply the *corpus* IDF (terms not in corpus get IDF = 1.0 by
#      convention — they carry no discriminative information).
#
# ══════════════════════════════════════════════════════════════════════════════

query_tokens   = tokenize(QUERY)
query_tf_raw   = Counter(query_tokens)
query_len      = len(query_tokens) if query_tokens else 1
query_tf       = {t: cnt / query_len for t, cnt in query_tf_raw.items()}

# Apply corpus IDF; unknown terms get idf = 1.0 (neutral)
query_tfidf: dict[str, float] = {
    t: tf_val * idf.get(t, 1.0)
    for t, tf_val in query_tf.items()
}

print(f"Query: \"{QUERY}\"")
print(f"\nQuery tokens after stopword removal: {query_tokens}")
print(f"\nQuery TF-IDF weights:")
for t, w in sorted(query_tfidf.items(), key=lambda x: -x[1]):
    print(f"  {t:<30}  {w:.6f}")


# ══════════════════════════════════════════════════════════════════════════════
#  STEP 8 – Rank documents by cosine similarity
# ══════════════════════════════════════════════════════════════════════════════

similarities: list[tuple[str, float]] = []
for doc_label, tfidf_row in zip(doc_labels, tfidf_matrix):
    sim = cosine_similarity(tfidf_row, query_tfidf)
    similarities.append((doc_label, sim))

# Sort descending by similarity score
similarities.sort(key=lambda x: -x[1])


# ══════════════════════════════════════════════════════════════════════════════
#  STEP 9 – Print results
# ══════════════════════════════════════════════════════════════════════════════

SEP = "=" * 78

print(f"\n{SEP}")
print("PART (1) — TF-IDF WEIGHT MATRIX  (top 10 terms per document shown)")
print(SEP)
print(f"\nFull matrix dimensions: {N} documents × {V} terms")
print(f"\n{'Document':<16} {'Top-10 highest-weighted terms (term: weight)'}")
print("-" * 78)
for doc_label, tfidf_row in zip(doc_labels, tfidf_matrix):
    top_terms = sorted(tfidf_row.items(), key=lambda x: -x[1])[:TOP_TF_IDF_TERMS]
    terms_str = "  |  ".join(f"{t}:{w:.4f}" for t, w in top_terms)
    print(f"{doc_label:<16} {terms_str}")

print(f"\n{SEP}")
print(f"PART (2) — DOCUMENT RANKING BY COSINE SIMILARITY TO QUERY")
print(SEP)
print(f'\nQuery : "{QUERY}"\n')
print(f"{'Rank':<6} {'Document':<16} {'Cosine Sim':>12}  {'Abstract snippet (first 120 chars)'}")
print("-" * 78)
for rank, (doc_label, sim) in enumerate(similarities, 1):
    doc_idx = int(doc_label.split("_")[1]) - 1
    snippet = raw_docs[doc_idx][:120].replace("\n", " ")
    marker  = " ◀ TOP" if rank <= 3 else ""
    print(f"{rank:<6} {doc_label:<16} {sim:>12.6f}  {snippet}…{marker}")
    if rank == TOP_K:
        remaining = [(lbl, s) for lbl, s in similarities[TOP_K:] if s > 0]
        print(f"       … {N - TOP_K} more documents "
              f"(of which {len(remaining)} have sim > 0) …")
        break

# ── Detailed view of the top-5 matches ──────────────────────────────────────
print(f"\n{SEP}")
print("TOP 5 MATCHES — FULL ABSTRACT")
print(SEP)
for rank, (doc_label, sim) in enumerate(similarities[:5], 1):
    doc_idx = int(doc_label.split("_")[1]) - 1
    print(f"\nRank {rank}  |  {doc_label}  |  cosine = {sim:.6f}")
    print(f"Title   : {df['title'].iloc[doc_idx]}")
    print(f"Abstract: {raw_docs[doc_idx]}")
    print("-" * 78)

# ── Summary statistics ───────────────────────────────────────────────────────
sims_nonzero = [s for _, s in similarities if s > 0]
print(f"\n{SEP}")
print("SIMILARITY STATISTICS")
print(SEP)
print(f"  Documents with cosine sim > 0  : {len(sims_nonzero)} / {N}")
print(f"  Max similarity                 : {similarities[0][1]:.6f}  ({similarities[0][0]})")
print(f"  Min non-zero similarity        : {sims_nonzero[-1]:.6f}  "
      f"({[lbl for lbl, s in similarities if s > 0][-1]})")
avg = sum(s for _, s in similarities) / N
print(f"  Mean similarity (all docs)     : {avg:.6f}")


# ══════════════════════════════════════════════════════════════════════════════
#  STEP 10 – Save outputs
# ══════════════════════════════════════════════════════════════════════════════

import os
OUT_DIR = "tfidf_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

# (a) Full TF-IDF matrix
tfidf_df.to_csv(os.path.join(OUT_DIR, "tfidf_matrix.csv"))
print(f"\nSaved: {OUT_DIR}/tfidf_matrix.csv  ({tfidf_df.shape[0]} rows × {tfidf_df.shape[1]} cols)")

# (b) Cosine similarity ranking
ranking_df = pd.DataFrame(
    [
        {
            "rank":          rank,
            "document":      doc_label,
            "cosine_sim":    round(sim, 6),
            "title":         df["title"].iloc[int(doc_label.split("_")[1]) - 1],
            "abstract_snippet": raw_docs[int(doc_label.split("_")[1]) - 1][:200],
        }
        for rank, (doc_label, sim) in enumerate(similarities, 1)
    ]
)
ranking_df.to_csv(os.path.join(OUT_DIR, "cosine_similarity_ranking.csv"), index=False)
print(f"Saved: {OUT_DIR}/cosine_similarity_ranking.csv")

# (c) IDF values for all terms
idf_df = pd.DataFrame(
    sorted(idf.items(), key=lambda x: x[1]),
    columns=["term", "idf"]
)
idf_df.to_csv(os.path.join(OUT_DIR, "idf_values.csv"), index=False)
print(f"Saved: {OUT_DIR}/idf_values.csv")

print("\nDone.")







Loaded 100 abstracts.

Vocabulary size : 4,616 unique terms
Building dense TF-IDF DataFrame …
TF-IDF matrix shape: (100, 4616)  (docs × terms)

Query: "deep learning model for accurate prediction and classification using neural network algorithms"

Query tokens after stopword removal: ['deep', 'learning', 'model', 'accurate', 'prediction', 'classification', 'neural', 'network', 'algorithms']

Query TF-IDF weights:
  accurate                        0.309101
  prediction                      0.302750
  classification                  0.302750
  neural                          0.291043
  deep                            0.285622
  algorithms                      0.270785
  network                         0.266249
  model                           0.200940
  learning                        0.116752

PART (1) — TF-IDF WEIGHT MATRIX  (top 10 terms per document shown)

Full matrix dimensions: 100 documents × 4616 terms

Document         Top-10 highest-weighted terms (term: weight)
------------

## Question 3 (25 points)

**Create your own word embedding model**

Use the data you collected for **Assignment 1** to build a word embedding model. You may use existing libraries (e.g., **gensim** or **transformers**) for training embeddings.

(1) Train a **300-dimensional** word embedding model (e.g., **Word2Vec, GloVe, ULMFiT, or a fine-tuned BERT model**).

(2) Visualize the embeddings using **PCA** or **t-SNE** in 2D. Create a scatter plot of at least **20 words** and show how similar words cluster together.

(3) Calculate the **cosine similarity** between a few pairs of words to examine whether the model captures semantic similarity accurately.

**References:**

- https://machinelearningmastery.com/develop-word-embeddings-python-gensim/
- https://jaketae.github.io/study/word2vec/


In [5]:
# Write your code here

# ═══════════════════════════════════════════════════════════════
#  CELL 1 — Run this first in Colab / Jupyter to install packages
# ═══════════════════════════════════════════════════════════════
import subprocess, sys

PACKAGES = ["gensim", "scikit-learn", "matplotlib", "pandas", "adjustText"]

for pkg in PACKAGES:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
    print(f"✓ {pkg}")

print("\nAll packages installed. Run the next cell now.")
"""
Word Embedding Model — Assignment Question 3
================================================================
(1) Train a 300-dimensional Word2Vec model on 100 abstracts.
(2) Visualize with PCA (2D) and t-SNE (2D) — scatter plots of
    50 selected words, colour-coded by semantic category.
(3) Compute pairwise cosine similarities for curated word pairs
    to verify the model captures semantic relationships.

Libraries: gensim (training), scikit-learn (PCA / t-SNE),
           matplotlib (plots).  All math from scratch where noted.
================================================================
HOW TO RUN:
  Google Colab : CSV_PATH = "/content/semantic_scholar_raw_abstracts.csv"
  Jupyter local: CSV_PATH = "semantic_scholar_raw_abstracts.csv"
"""

import re, os, math
from collections import Counter
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")           # headless — no display needed
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec

from gensim.models import Word2Vec
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# optional: prettier label placement
try:
    from adjustText import adjust_text
    HAS_ADJUST = True
except ImportError:
    HAS_ADJUST = False

# ── CONFIG ────────────────────────────────────────────────────────────────────
CSV_PATH   = "semantic_scholar_raw_abstracts.csv"   # ← change for your env
OUT_DIR    = "embedding_outputs"
VECTOR_DIM = 300          # embedding dimensionality
WINDOW     = 5            # context window
MIN_COUNT  = 2            # ignore words appearing < 2 times
EPOCHS     = 100          # training epochs (more = better for small corpus)
SEED       = 42
# ─────────────────────────────────────────────────────────────────────────────

os.makedirs(OUT_DIR, exist_ok=True)

STOPWORDS = {
    "a","an","the","and","or","but","in","on","at","to","for","of","with",
    "by","from","as","is","are","was","were","be","been","being","have",
    "has","had","that","this","these","those","it","its","we","our","they",
    "their","which","also","can","may","will","more","not","no","into",
    "than","each","all","both","other","such","after","before","up","out",
    "over","under","between","through","about","while","when","where",
    "what","if","so","do","does","did","he","she","his","her","i","you",
    "your","them","us","my","me","who","there","here","any","some","most",
    "very","just","only","then","nor","via","per","et","al","de","en","doi",
    "using","used","use","uses","paper","study","work","results","result",
    "based","proposed","approach","system","two","three","one","new","well",
}

# ══════════════════════════════════════════════════════════════════════════════
# 0. Load & preprocess
# ══════════════════════════════════════════════════════════════════════════════

df = (pd.read_csv(CSV_PATH)
        .pipe(lambda d: d[d["abstract"].notna()])
        .head(100)
        .reset_index(drop=True))

raw_docs = df["abstract"].tolist()
print(f"Loaded {len(raw_docs)} abstracts.")

def tokenize(text: str) -> list[str]:
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    return [t for t in text.split() if t not in STOPWORDS and len(t) > 2]

sentences: list[list[str]] = [tokenize(doc) for doc in raw_docs]

# Vocabulary stats
all_tokens = [t for s in sentences for t in s]
vocab_freq  = Counter(all_tokens)
print(f"Total tokens  : {len(all_tokens):,}")
print(f"Unique tokens : {len(vocab_freq):,}")
print(f"Top 10 terms  : {vocab_freq.most_common(10)}\n")

# ══════════════════════════════════════════════════════════════════════════════
# 1. Train 300-D Word2Vec (Skip-gram)
# ══════════════════════════════════════════════════════════════════════════════

print("Training Word2Vec (skip-gram, 300-D) …")
model = Word2Vec(
    sentences  = sentences,
    vector_size= VECTOR_DIM,
    window     = WINDOW,
    min_count  = MIN_COUNT,
    workers    = 4,
    sg         = 1,           # skip-gram
    hs         = 0,           # negative sampling
    negative   = 10,
    epochs     = EPOCHS,
    seed       = SEED,
)

wv = model.wv
print(f"Vocabulary in model : {len(wv):,} words")
print(f"Vector dimensions   : {wv.vector_size}\n")

# Save model
model.save(os.path.join(OUT_DIR, "word2vec_300d.model"))
print(f"Model saved → {OUT_DIR}/word2vec_300d.model\n")

# Quick sanity: most similar to "learning"
for probe in ["learning", "network", "prediction", "cancer"]:
    if probe in wv:
        sims = wv.most_similar(probe, topn=5)
        print(f"  most_similar('{probe}') → {[w for w,_ in sims]}")
print()

# ══════════════════════════════════════════════════════════════════════════════
# 2. Visualise — PCA + t-SNE
# ══════════════════════════════════════════════════════════════════════════════

# ── Curated word groups (≥20 words for scatter, colour-coded by theme) ───────
WORD_GROUPS = {
    "ML / DL":          ["learning", "neural", "network", "deep", "training",
                         "classification", "prediction", "model", "algorithm",
                         "convolutional"],
    "Data & Features":  ["data", "features", "dataset", "analysis", "feature",
                         "accuracy", "performance", "detection", "evaluation"],
    "Medical / Bio":    ["disease", "cancer", "patient", "clinical", "health",
                         "diagnosis", "medical", "brain"],
    "NLP / Text":       ["text", "language", "information", "knowledge",
                         "detection", "recognition"],
    "Systems & Infra":  ["network", "cloud", "sensor", "energy", "wireless",
                         "computing"],
}

# Flatten, keep only words in model, de-duplicate
seen = set()
plot_words, plot_groups = [], []
GROUP_COLORS = {
    "ML / DL":          "#4e79a7",
    "Data & Features":  "#f28e2b",
    "Medical / Bio":    "#e15759",
    "NLP / Text":       "#76b7b2",
    "Systems & Infra":  "#59a14f",
}

for group, words in WORD_GROUPS.items():
    for w in words:
        if w in wv and w not in seen:
            plot_words.append(w)
            plot_groups.append(group)
            seen.add(w)

print(f"Words available for plot: {len(plot_words)}")
assert len(plot_words) >= 20, "Need at least 20 words — check vocabulary"

vectors_300d = np.array([wv[w] for w in plot_words])   # (n, 300)
colors       = [GROUP_COLORS[g] for g in plot_groups]

# ── PCA 2-D ──────────────────────────────────────────────────────────────────
pca = PCA(n_components=2, random_state=SEED)
coords_pca = pca.fit_transform(vectors_300d)            # (n, 2)
var_explained = pca.explained_variance_ratio_ * 100

# ── t-SNE 2-D ────────────────────────────────────────────────────────────────
# perplexity must be < n_samples
perp = min(15, len(plot_words) - 1)
tsne = TSNE(n_components=2, perplexity=perp, max_iter=2000,
            random_state=SEED, init="pca", learning_rate="auto")
coords_tsne = tsne.fit_transform(vectors_300d)          # (n, 2)

# ── Build the figure ──────────────────────────────────────────────────────────
plt.rcParams.update({
    "font.family":      "DejaVu Sans",
    "axes.spines.top":  False,
    "axes.spines.right":False,
    "figure.facecolor": "#0f1117",
    "axes.facecolor":   "#161b22",
    "axes.labelcolor":  "#e6edf3",
    "xtick.color":      "#8b949e",
    "ytick.color":      "#8b949e",
    "text.color":       "#e6edf3",
    "grid.color":       "#21262d",
    "grid.linewidth":   0.5,
})

fig = plt.figure(figsize=(20, 10), dpi=140)
gs  = GridSpec(1, 2, figure=fig, wspace=0.08)

legend_handles = [
    mpatches.Patch(color=c, label=g)
    for g, c in GROUP_COLORS.items()
    if g in set(plot_groups)
]

def draw_panel(ax, coords, title, subtitle):
    """Render one scatter panel with labelled points."""
    ax.set_title(title, fontsize=15, fontweight="bold",
                 color="#e6edf3", pad=10)
    ax.set_xlabel(subtitle, fontsize=9, color="#8b949e")
    ax.grid(True, linestyle="--", alpha=0.4)

    # glow effect: draw large faint dots first, then crisp dots on top
    ax.scatter(coords[:, 0], coords[:, 1],
               c=colors, s=200, alpha=0.18, linewidths=0)
    sc = ax.scatter(coords[:, 0], coords[:, 1],
                    c=colors, s=70, alpha=0.95,
                    linewidths=0.6, edgecolors="white", zorder=3)

    # labels
    texts = []
    for i, word in enumerate(plot_words):
        t = ax.annotate(
            word,
            (coords[i, 0], coords[i, 1]),
            fontsize=7.5,
            color="#c9d1d9",
            xytext=(4, 4),
            textcoords="offset points",
        )
        texts.append(t)

    if HAS_ADJUST:
        adjust_text(texts, ax=ax,
                    arrowprops=dict(arrowstyle="-", color="#444c56", lw=0.5),
                    expand_points=(1.4, 1.4))
    return sc

ax1 = fig.add_subplot(gs[0])
draw_panel(ax1, coords_pca,
           "PCA  —  300-D → 2-D",
           f"PC1 ({var_explained[0]:.1f}% var)  ·  PC2 ({var_explained[1]:.1f}% var)")

ax2 = fig.add_subplot(gs[1])
draw_panel(ax2, coords_tsne,
           "t-SNE  —  300-D → 2-D",
           f"perplexity={perp}  ·  2000 iterations")

fig.legend(handles=legend_handles, loc="lower center",
           ncol=len(legend_handles), fontsize=9,
           frameon=True, facecolor="#161b22", edgecolor="#30363d",
           labelcolor="#c9d1d9", markerscale=1.2,
           bbox_to_anchor=(0.5, -0.02))

fig.suptitle("Word Embeddings — Word2Vec (Skip-gram, 300-D)  ·  Semantic Scholar Abstracts",
             fontsize=14, fontweight="bold", color="#e6edf3", y=1.01)

plot_path = os.path.join(OUT_DIR, "embedding_visualization.png")
plt.tight_layout()
plt.savefig(plot_path, bbox_inches="tight",
            facecolor=fig.get_facecolor(), dpi=140)
plt.close()
print(f"Scatter plot saved → {plot_path}\n")

# ══════════════════════════════════════════════════════════════════════════════
# 3. Cosine Similarity — from scratch
# ══════════════════════════════════════════════════════════════════════════════

def cosine_similarity_scratch(vec_a: np.ndarray,
                               vec_b: np.ndarray) -> float:
    """
    cosine_sim(a, b) = dot(a, b) / (||a|| * ||b||)
    Implemented entirely from scratch using Python math.
    """
    dot   = float(np.dot(vec_a, vec_b))        # Σ aᵢbᵢ
    norm_a = math.sqrt(float(np.dot(vec_a, vec_a)))   # √(Σ aᵢ²)
    norm_b = math.sqrt(float(np.dot(vec_b, vec_b)))
    if norm_a == 0.0 or norm_b == 0.0:
        return 0.0
    return dot / (norm_a * norm_b)

# Word pairs: (word_a, word_b, expected_relationship)
PAIRS = [
    # Highly related — should score HIGH
    ("learning",      "training",    "both core ML concepts"),
    ("neural",        "network",     "collocate strongly in corpus"),
    ("classification","prediction",  "related supervised-learning tasks"),
    ("cancer",        "disease",     "medical domain siblings"),
    ("deep",          "convolutional","both describe neural arch types"),
    ("data",          "dataset",     "singular / collective form"),
    ("algorithm",     "model",       "both describe ML artefacts"),
    ("sensor",        "wireless",    "IoT / embedded systems"),
    # Moderately related — should score MEDIUM
    ("detection",     "accuracy",    "evaluation of detection quality"),
    ("network",       "cloud",       "computing infrastructure"),
    ("health",        "patient",     "clinical context"),
    ("text",          "language",    "NLP domain"),
    # Weakly / un-related — should score LOW
    ("learning",      "cancer",      "ML vs medical — different domains"),
    ("neural",        "cloud",       "DL vs infra — different domains"),
    ("energy",        "language",    "energy systems vs NLP — unrelated"),
    ("wireless",      "diagnosis",   "systems vs medical — unrelated"),
]

print("=" * 72)
print("PART (3) — COSINE SIMILARITY BETWEEN WORD PAIRS  (from scratch)")
print("=" * 72)
print(f"\n{'Word A':<18} {'Word B':<18} {'Cosine Sim':>11}  "
      f"{'Gensim Sim':>10}  Relationship")
print("-" * 72)

rows = []
for wa, wb, note in PAIRS:
    in_model = wa in wv and wb in wv
    if in_model:
        sim_scratch = cosine_similarity_scratch(wv[wa], wv[wb])
        sim_gensim  = float(wv.similarity(wa, wb))
        delta       = abs(sim_scratch - sim_gensim)
        rows.append((wa, wb, sim_scratch, sim_gensim, note))
        print(f"{wa:<18} {wb:<18} {sim_scratch:>11.6f}  "
              f"{sim_gensim:>10.6f}  {note}")
        assert delta < 1e-5, f"Mismatch! scratch={sim_scratch}, gensim={sim_gensim}"
    else:
        missing = [w for w in (wa, wb) if w not in wv]
        print(f"{wa:<18} {wb:<18}  {'N/A (not in vocab)':>23}  "
              f"missing: {missing}")

print(f"\n✓ All scratch cosine values match gensim (delta < 1e-5)")

# ── Bar chart of similarities ─────────────────────────────────────────────────
if rows:
    fig2, ax = plt.subplots(figsize=(13, 6), dpi=130)
    fig2.patch.set_facecolor("#0f1117")
    ax.set_facecolor("#161b22")

    labels = [f"{r[0]} ↔ {r[1]}" for r in rows]
    sims   = [r[2] for r in rows]
    notes  = [r[4] for r in rows]

    # colour by expected band
    HIGH_COL, MID_COL, LOW_COL = "#4e79a7", "#f28e2b", "#e15759"
    bar_colors = []
    for note in notes:
        if   "HIGH" in note.upper() or note.startswith("both core") or "strongly" in note:
            bar_colors.append(HIGH_COL)
        elif "LOW" in note.upper() or "unrelated" in note.lower():
            bar_colors.append(LOW_COL)
        else:
            bar_colors.append(MID_COL)
    # fallback: use value thresholds
    bar_colors = [
        HIGH_COL if s >= 0.5 else (MID_COL if s >= 0.25 else LOW_COL)
        for s in sims
    ]

    bars = ax.barh(labels[::-1], sims[::-1], color=bar_colors[::-1],
                   height=0.65, edgecolor="#21262d", linewidth=0.4)
    for bar, val in zip(bars, sims[::-1]):
        ax.text(val + 0.01, bar.get_y() + bar.get_height() / 2,
                f"{val:.4f}", va="center", ha="left",
                fontsize=8, color="#c9d1d9")

    ax.set_xlim(0, 1.05)
    ax.set_xlabel("Cosine Similarity", color="#8b949e", fontsize=10)
    ax.set_title("Cosine Similarity between Word Pairs\n"
                 "Word2Vec 300-D  ·  Semantic Scholar Abstracts",
                 fontsize=12, fontweight="bold", color="#e6edf3", pad=10)
    ax.tick_params(colors="#8b949e")
    ax.spines[["top","right","bottom"]].set_visible(False)
    ax.spines["left"].set_color("#30363d")
    ax.grid(axis="x", linestyle="--", alpha=0.3, color="#21262d")
    ax.axvline(0.5,  color=HIGH_COL, lw=0.8, linestyle=":", alpha=0.6)
    ax.axvline(0.25, color=MID_COL,  lw=0.8, linestyle=":", alpha=0.6)

    legend_elems = [
        mpatches.Patch(color=HIGH_COL, label="High similarity (≥ 0.50)"),
        mpatches.Patch(color=MID_COL,  label="Medium similarity (0.25–0.50)"),
        mpatches.Patch(color=LOW_COL,  label="Low similarity (< 0.25)"),
    ]
    ax.legend(handles=legend_elems, loc="lower right",
              fontsize=8, frameon=True, facecolor="#161b22",
              edgecolor="#30363d", labelcolor="#c9d1d9")

    bar_path = os.path.join(OUT_DIR, "cosine_similarity_barchart.png")
    plt.tight_layout()
    plt.savefig(bar_path, bbox_inches="tight",
                facecolor=fig2.get_facecolor(), dpi=130)
    plt.close()
    print(f"\nBar chart saved → {bar_path}")

# ══════════════════════════════════════════════════════════════════════════════
# 4. Nearest-neighbour table (bonus)
# ══════════════════════════════════════════════════════════════════════════════

PROBE_WORDS = ["learning", "network", "prediction", "cancer",
               "deep", "data", "detection", "classification"]

print("\n" + "=" * 72)
print("BONUS — TOP-5 NEAREST NEIGHBOURS (gensim cosine)")
print("=" * 72)
for probe in PROBE_WORDS:
    if probe in wv:
        neighbours = wv.most_similar(probe, topn=5)
        nbr_str = "  |  ".join(f"{w} ({s:.3f})" for w, s in neighbours)
        print(f"  {probe:<18} →  {nbr_str}")

# ══════════════════════════════════════════════════════════════════════════════
# 5. Save similarity table to CSV
# ══════════════════════════════════════════════════════════════════════════════

sim_df = pd.DataFrame([
    {"word_a": wa, "word_b": wb,
     "cosine_similarity": round(sim, 6),
     "relationship": note}
    for wa, wb, sim, _, note in rows
])
sim_csv = os.path.join(OUT_DIR, "cosine_similarity_pairs.csv")
sim_df.to_csv(sim_csv, index=False)
print(f"\nSimilarity table saved → {sim_csv}")
print("\nDone.")



✓ gensim
✓ scikit-learn
✓ matplotlib
✓ pandas
✓ adjustText

All packages installed. Run the next cell now.
Loaded 100 abstracts.
Total tokens  : 14,514
Unique tokens : 4,648
Top 10 terms  : [('learning', 178), ('data', 162), ('manual', 152), ('machine', 145), ('model', 116), ('models', 76), ('accuracy', 58), ('network', 56), ('time', 50), ('analysis', 49)]

Training Word2Vec (skip-gram, 300-D) …
Vocabulary in model : 2,190 words
Vector dimensions   : 300

Model saved → embedding_outputs/word2vec_300d.model

  most_similar('learning') → ['machine', 'employing', 'extreme', 'multilayer', 'elm']
  most_similar('network') → ['traffic', 'multimedia', 'anaconda', 'iterative', 'abnormal']
  most_similar('prediction') → ['cyanobacterial', 'bloom', 'blooms', 'outbreak', 'yield']
  most_similar('cancer') → ['variant', 'prostate', 'ovarian', 'missense', 'ggs']

Words available for plot: 37


/tmp/ipykernel_400/1284309976.py:270: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


Scatter plot saved → embedding_outputs/embedding_visualization.png

PART (3) — COSINE SIMILARITY BETWEEN WORD PAIRS  (from scratch)

Word A             Word B              Cosine Sim  Gensim Sim  Relationship
------------------------------------------------------------------------
learning           training              0.102730    0.102730  both core ML concepts
neural             network               0.335795    0.335795  collocate strongly in corpus
classification     prediction            0.177114    0.177114  related supervised-learning tasks
cancer             disease               0.223403    0.223403  medical domain siblings
deep               convolutional         0.406615    0.406615  both describe neural arch types
data               dataset               0.125354    0.125354  singular / collective form
algorithm          model                 0.185282    0.185282  both describe ML artefacts
sensor             wireless              0.624062    0.624062  IoT / embedded syst

## Question 4 (20 Points)

**Create your own training and evaluation dataset for an NLP task.**

**You do not need to write a program for this question.**

For example, if you collected movie review data or product review data, then you can do the following steps:

* Read each review (abstract or tweet) you collected in detail, and annotate each review with a sentiment (**positive, negative, or neutral**).

* Save the annotated dataset into a **CSV** file with three columns (`document_id`, `clean_text`, `sentiment`), upload the CSV file to GitHub, and submit the file link below.

This dataset will be used for **Assignment 4: Sentiment Analysis and Text Classification**.


1. Which NLP task would you like to perform on your selected dataset (**NER, summarization, sentiment analysis, or text classification**)?
2. Explain the labeling schema you used and mention the labels.

3. You may use AI assistance for labeling the data only.


In [ ]:
# The GitHub link of your final csv file


# Link:https://github.com/SRIJANRAOS/srijanraos_INFO5731_spring2026/blob/main/sentiment_dataset.csv



# Mandatory Question (5 Points)

Provide your thoughts on the assignment by filling this survey link. What did you find challenging, and what aspects did you enjoy? Your opinion on the provided time to complete the assignment.

In [6]:
I found this assignment to be a valuable learning experience, especially in understanding how to work with real-world text data and apply NLP techniques. One of the aspects I enjoyed the most was the hands-on component, particularly data cleaning and preprocessing, as it gave me a clearer idea of how raw data is transformed into a usable format for analysis.

The most challenging part for me was dealing with technical issues and debugging errors while running the code, especially when working with external tools like APIs or handling file paths. At times, setting up the environment and ensuring everything worked correctly took longer than expected. Additionally, understanding some of the concepts in depth while implementing them required extra effort.

Regarding the time provided, I feel it was generally sufficient, but slightly more time would have been helpful to troubleshoot technical issues and refine the final output. Overall, the assignment was engaging and helped strengthen both my practical coding skills and conceptual understanding of NLP.

SyntaxError: invalid syntax (3549640082.py, line 1)